# FinCIMM / FAB-CI Experimental Notebook

This is a cleaned, output-stripped version of the original experimental notebook. 
Update local dataset/image paths before running. The public repository intentionally includes only sample/anonymized data.


In [ ]:
%pip install -r ../requirements.txt


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np

dataset = pd.read_excel("/content/drive/MyDrive/NLP/test/credit_card/credit_card.xlsx")
#dataset = dataset.iloc[0:1096,:]
dataset.columns

In [ ]:
import os
import torch
import torchvision
import torch.nn as nn
import torch.optim as optim
from PIL import Image
from tqdm import tqdm
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from transformers import RobertaModel, RobertaTokenizer
from sklearn.metrics import f1_score, accuracy_score, classification_report
#from src.loss_functions.losses import AsymmetricLoss

import re
import json
import random
import shutil
import argparse
import functools
import contextlib
import numpy as np
import pandas as pd
from collections import Counter

import time
import logging
from datetime import timedelta

os.environ['CUDA_LAUNCH_BLOCKING'] = "1"

In [ ]:
batch_sz = 8
drop_img_percent=0.0
dropout=0.1
embed_sz=300
freeze_img=0
freeze_txt=0
gradient_accumulation_steps=24
hidden=[]
hidden_sz=768
img_embed_pool_type="avg"
img_hidden_sz=2048
include_bn=True
lr=1e-4
lr_factor=0.5
lr_patience=2
max_epochs=10
max_seq_len=512 #changed it
model_name="mmbt"
n_workers=2
name="nameless"
num_image_embeds=1
patience=10
savedir="/content/drive/MyDrive/NLP/test/credit_card/"
seed=42
task="mmimdb"
task_type= "multilabel"#, "classification"]
warmup=0.1
weight_classes=1
n_classes = 10

tokenizer = RobertaTokenizer.from_pretrained("roberta-base", return_dict=True, do_lower_case=True)

In [ ]:
dataset.columns

In [ ]:
dataset.drop(columns = ['Over-all_Complaint Label', 'Sentiment','Emotion','Severity level'], inplace = True)

In [ ]:
def check_aspect(aspect,j):
  aspects = []
  for i in [0,1,2]:
    val = aspect[i][j]
    if(str(val) == 'nan'):
      aspects.append('no_aspect')
    if val in [' feedback',' Feedback','fraud\n','Feedaback','General opinion','Genera review','General review','Feedbck','Feeedback','Feedabck','Feedback','review','freedback','feeedback','feedbck','feedaback','fedback','feedabck','feedback','personal opinion','personal opinion,','general opinion','advise','genera review','general review']:
      aspects.append('review')
    elif val in ['miscellaneous information\n',' miscellaneous information\n','miscellaneous Information\n','miscellaneous Information','MIscellaneous information','Miscellaneous inforamation','Miscellaneus information',' Miscellaneous information','Miscellaneous information, inflation','Miscellaneous inforation','Misceklaneous information','Miscelaneous information','Miscellaneous iformation','Miscellaneous iformation','Miscelllaneous information','Misceallneous information',' miscellaneous information','Miscellaneous information','miscelllaneous information','miscellaneous inforamation','miscellaneous inforation','misceallneous information','miscellaneous information, inflation','misellaneous infromation','miscelleneos information','miscellaneous information','miscellaneous','miscellameous information','miscellaneus information','misellaneous information','miscelaneous information','misceklaneous information','miscellaneous iformation','misceallaneous information','financial advertisement','financial advertiement','financial advertisemnt']:
      aspects.append('misc_info')
    elif val in ['delay response, ','delay response ','delay response\n','Delay Response ','Delay response','Delay Response',' delay response','delay, response','delay resoponse','delay respose','delay response','dealy response','delay resonse','delay response,','delay reponse','apologise']:
      aspects.append('provider_response')
    elif val in ['payment \n',' bank issue','payment failure\n','Technical issue','payment\n','Bank Issue','banking  ','payment failure, miscellaneous information\n','Refund issue',' payment failure',' payment issue','Bank issue','banking ','Payment ',' payment ',' payment','Payment','Banking',' banking','payment ','payment','paymet','payent issue','payment issue','payment issue, bank issue','payment failure','payment failure, miscellaneous information','banking','bankinhg','bank issue','net banking','bank issue, harassment','internet banking','refund issue']:
      aspects.append('net_banking_issue')
    elif val in ['fraud ',' fraud ',' harassment',' fruad',' harrashment','Harassment','harassment ','Harrasment','Fraud',' fraud','haraasment','harrasment','harassment','harrashment','fruad','fraud','manipulation']:
      aspects.append('consumer_safety')
    elif val in ['Financial Policy','trade balance','news','dept collection','debt collection','exports']:
      aspects.append('financial_info')
    elif val in ['crdentials ',' credentials issue','cedentials ',' technicality','Credential related','Credentials','Technical Issue',' credentials ',' credentials','credentials ','crdentials','credentials','credentials issue','credential','cedentials','credential related','technicality','techinicality','technical issue']:
      aspects.append('credential_error')
    elif val in ['Financial Advertisement',' inflation','Financial Advertisemnt',' debt collection','Finacial policy','Financial policies',' Financial Policies','Financial Policies','Financial policy','Financial advertisement','finacial policy','financial policy','financial policies','financial poilicy','financial stability','financial gain','inflation']:
      aspects.append('financial_situation')
    elif val in ['news ',' news','Genaral query','general Query','Genearl query',' news ','Genral query',' General query ','General query ','General Query ',' General Query ','General Query','General query','general_query','genearl query','general qurey','general query','genral query','genaral query','general qurery','tax increases','equal taxation']:
      aspects.append('general_query')
    else:
      if val in [' ']:
        aspects.append('no_aspect')
  return aspects


def get_labels(df):
    aspect = [list(dataset['Aspects_1']), list(dataset['Aspect_2']), list(dataset['Aspect_3'])]
    aspects = []

    for i in range(df.shape[0]):
        aspects.append(check_aspect(aspect,i))

    return aspects

aspect_labels = get_labels(dataset)
print(aspect_labels)
print(len(aspect_labels))

In [ ]:
def get_title_review_comb(X_reviews):
    title_review_comb = []
    for i in range(X_reviews.shape[0]):
        if(str(type(X_reviews.iloc[i,1])) == "<class 'str'>"):
          title_review_comb.append(str(X_reviews.iloc[i,1])+str(X_reviews.iloc[i,2]))
        else:
          # print(i)
          title_review_comb.append(X_reviews.iloc[i,2])

    text_reviews = []
    for i in range(X_reviews.shape[0]):
        words = re.split(r'\W+', title_review_comb[i])

        words = [word.lower() for word in words]
        text = ' '.join(words)
        words = text.split()
        text = ' '.join(words)
        text_reviews.append(text)

    return text_reviews

# def check_aspect_comp(aspect,complaint_label,j):
#   aspects = []
#   comp_labels = []
#   for i in [0,1,2]:
#     val1 = aspect[i][j]
#     val2 = complaint_label[i][j]
#     if(str(val1) == 'nan') or val1 == ' ':
#       continue
#     elif(val1 == 'fabric'):
#       aspects.append('quality')
#     else:
#       aspects.append(val1)
#     comp_labels.append(val2)

#   return aspects,comp_labels

def get_web_entities(dataset):
    web_entities = []
    for mul_entities in list(dataset['Web Entity']):
      lables_list = (re.sub("[^a-zA-Z0-9,.)]", " ", mul_entities)).split(',')
      # print(lables_list)
      lables_list_mod = []
      for label in lables_list:
        label_mod = " ".join(label.split())
        # print(label_mod)
        if(label_mod == ''):
          continue
        lables_list_mod.append(label_mod.lower())

      web_entities.append(lables_list_mod)

    return web_entities


text_review = get_title_review_comb(dataset)
web_entities = get_web_entities(dataset)
#aspect_labels= get_labels(dataset)

#print(len(aspect_labels))
#print(len(complaint_labels))
print(len(text_review))
print(text_review)
print(len(web_entities))
print(web_entities)
#print(len(images_path))

In [ ]:
# dataset.columns

In [ ]:
drive_path = "/content/drive/MyDrive/NLP/BART" #change the file path where images are stored
images_path = []
for i in range(dataset.shape[0]):
        txt = dataset.iloc[i][13]
        x = txt.split("/")
        images_path.append(drive_path + "/" + x[-1])
print(len(images_path))
print(images_path)

In [ ]:
# import os

# # List of image file paths
# image_paths = images_path

# # Directory where you want to store the images with new names
# output_directory = "/content/drive/MyDrive/harsha_451/new_2"
# i = 1
# # Create the output directory if it doesn't exist
# if not os.path.exists(output_directory):
#     os.makedirs(output_directory)

# # Iterate through the image paths
# for image in image_paths:
#     # Get the filename and extension
#     file_name = os.path.basename(image)
#     file_name, file_extension = os.path.splitext(file_name)

#     # Generate a new file name (you can change this as per your needs)
#     new_file_name = str(i) + file_extension

#     # Create the new file path in the output directory
#     new_image_path = os.path.join(output_directory, new_file_name)

#     # Copy the image to the new location with the new name
#     shutil.copy(image, new_image_path)
#     print(i)
#     i = i+1
#     #print(f"Renamed and moved '{image}' to '{new_image_path}'")

# print("All images have been renamed and stored in the same folder.")



In [ ]:
# imgs_path = '/content/drive/MyDrive/harsha_451/new_final_images/'
# dir = os.listdir(imgs_path)
# dir.sort()
# directory = dir
# for i in range(len(directory)):
#   directory[i] = imgs_path+directory[i]

# images_path_new = directory
# print(len(images_path_new))

In [ ]:
# First Split for Train and Test
text_train,text_test, web_ent_train,web_ent_test, img_train,img_test, y1_train,y1_test  = train_test_split(np.array(text_review), np.array(web_entities), np.array(images_path),
                                                                                                           np.array(aspect_labels), test_size=0.1, random_state=seed, shuffle=True)
# Next split Train in to training and validation
text_tr,text_val, web_ent_tr,web_ent_val, img_tr,img_val, y1_tr,y1_val = train_test_split(text_train, web_ent_train, img_train, y1_train, test_size=0.2, random_state = seed, shuffle=True)

# print(web_ent_tr.shape)
# print(web_ent_test.shape)
# print(web_ent_val.shape)

## Defining Functions and Classes

## Defining Functions and Classes

#### Classes and Utility functions


In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def save_checkpoint(state, is_best, checkpoint_path, filename="checkpoint.pt"):
    filename = os.path.join(checkpoint_path, filename)
    torch.save(state, filename)
    if is_best:
        shutil.copyfile(filename, os.path.join(checkpoint_path, "model_best.pt"))


def load_checkpoint(model, path):
    best_checkpoint = torch.load(path)
    model.load_state_dict(best_checkpoint["state_dict"])


def truncate_seq_pair(tokens_a, tokens_b, max_length):
    """Truncates a sequence pair in place to the maximum length.
    Copied from https://github.com/huggingface/pytorch-pretrained-BERT
    """
    while True:
        total_length = len(tokens_a) + len(tokens_b)
        if total_length <= max_length:
            break
        if len(tokens_a) > len(tokens_b):
            tokens_a.pop()
        else:
            tokens_b.pop()


def log_metrics(set_name, metrics, logger):
        logger.info(
            "{}: Loss_mean_overall: {:.5f} | Macro F1 {:.5f} | Micro F1: {:.5f}".format(
                set_name, metrics["loss_mean_overall"], metrics["macro_f1"], metrics["micro_f1"]
            )
        )


@contextlib.contextmanager
def numpy_seed(seed, *addl_seeds):
    """Context manager which seeds the NumPy PRNG with the specified seed and
    restores the state afterward"""
    if seed is None:
        yield
        return
    if len(addl_seeds) > 0:
        seed = int(hash((seed, *addl_seeds)) % 1e6)
    state = np.random.get_state()
    np.random.seed(seed)
    try:
        yield
    finally:
        np.random.set_state(state)

def get_labels_and_frequencies(aspects):
    label_freqs = Counter()
    data_labels = aspects
    if type(data_labels[0]) == list:
        for label_row in data_labels:
            label_freqs.update(label_row)
    else:
        label_freqs.update(data_labels)

    return list(label_freqs.keys()), label_freqs

aspect_label, aspect_label_freqs = get_labels_and_frequencies(aspect_labels)
print(aspect_label_freqs)
print(len(aspect_label))

In [ ]:
class AsymmetricLoss(nn.Module):
    def __init__(self, gamma_neg=4, gamma_pos=1, clip=0.05, eps=1e-8, disable_torch_grad_focal_loss=True):
        super(AsymmetricLoss, self).__init__()

        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip = clip
        self.disable_torch_grad_focal_loss = disable_torch_grad_focal_loss
        self.eps = eps

    def forward(self, x, y):
        """"
        Parameters
        ----------
        x: input logits
        y: targets (multi-label binarized vector)
        """

        # Calculating Probabilities
        x_sigmoid = torch.sigmoid(x)
        xs_pos = x_sigmoid
        xs_neg = 1 - x_sigmoid

        # Asymmetric Clipping
        if self.clip is not None and self.clip > 0:
            xs_neg = (xs_neg + self.clip).clamp(max=1)

        # Basic CE calculation
        los_pos = y * torch.log(xs_pos.clamp(min=self.eps))
        los_neg = (1 - y) * torch.log(xs_neg.clamp(min=self.eps))
        loss = los_pos + los_neg

        # Asymmetric Focusing
        if self.gamma_neg > 0 or self.gamma_pos > 0:
            if self.disable_torch_grad_focal_loss:
                torch.set_grad_enabled(False)
            pt0 = xs_pos * y
            pt1 = xs_neg * (1 - y)  # pt = p if t > 0 else 1-p
            pt = pt0 + pt1
            one_sided_gamma = self.gamma_pos * y + self.gamma_neg * (1 - y)
            one_sided_w = torch.pow(1 - pt, one_sided_gamma)
            if self.disable_torch_grad_focal_loss:
                torch.set_grad_enabled(True)
            loss *= one_sided_w

        return -loss.sum()


In [ ]:
class JsonlDataset(Dataset):
    def __init__(self, reviews, web_entities, images_path, labels, tokenizer, transforms):
        self.text_data = reviews
        self.web_entities = web_entities
        self.img_data = images_path
        self.aspect_data = labels
        self.tokenizer = tokenizer

        self.text_start_token = ["<s>"] if model_name != "mmbt" else ["</s>"]

        self.max_seq_len = max_seq_len - num_image_embeds

        self.transforms = transforms

    def __len__(self):
        return len(self.text_data)

    def __getitem__(self, index):
        sent1 = self.tokenizer.tokenize(self.text_data[index])
        sent2 = self.tokenizer.tokenize(" ".join(self.web_entities[index]))
        truncate_seq_pair(sent1, sent2, self.max_seq_len - 3)

        sentence = self.text_start_token + sent1 + ["</s>"] + sent2 + ["</s>"]
        segment = torch.cat(
                [torch.zeros(2 + len(sent1)), torch.ones(len(sent2) + 1)]
            )

        sentence = torch.LongTensor(self.tokenizer.convert_tokens_to_ids(sentence))

        label = torch.zeros(10)
        label[
              [aspect_label.index(tgt) for tgt in self.aspect_data[index]]
              # [aspect_label.index(y1_tr[index])]
            ] = 1

        image = None
        if self.img_data[index]:
            image = Image.open(self.img_data[index]).convert("RGB")
        else:
            image = Image.fromarray(128 * np.ones((256, 256, 3), dtype=np.uint8))
        image = self.transforms(image)

        # The first SEP is part of Image Token.
        segment = segment[1:]
        sentence = sentence[1:]
        # The first segment (0) is of images.
        segment += 1

        return sentence, segment, image, label

# class JsonlDataset(Dataset):
#     def __init__(self, reviews, images_path, labels, tokenizer, transforms):
#         self.text_data = reviews
#         self.img_data = images_path
#         self.aspect_data = labels
#         self.tokenizer = tokenizer

#         self.text_start_token = ["<s>"] if model_name != "mmbt" else ["</s>"]

#         self.max_seq_len = max_seq_len - num_image_embeds

#         self.transforms = transforms

#     def __len__(self):
#         return len(self.text_data)

#     def __getitem__(self, index):
#         sent1 = self.tokenizer.tokenize(self.text_data[index])
#         sent2 = self.tokenizer.tokenize(" ")
#         truncate_seq_pair(sent1, sent2, self.max_seq_len - 3)

#         sentence = self.text_start_token + sent1 + ["</s>"] + sent2 + ["</s>"]
#         segment = torch.cat(
#                 [torch.zeros(2 + len(sent1)), torch.ones(len(sent2) + 1)]
#             )

#         sentence = torch.LongTensor(self.tokenizer.convert_tokens_to_ids(sentence))

#         label = torch.zeros(7)
#         label[
#               [aspect_label.index(tgt) for tgt in self.aspect_data[index]]
#             ] = 1

#         image = None
#         if self.img_data[index]:
#             image = Image.open(self.img_data[index]).convert("RGB")
#         else:
#             image = Image.fromarray(128 * np.ones((256, 256, 3), dtype=np.uint8))
#         image = self.transforms(image)

#         # The first SEP is part of Image Token.
#         segment = segment[1:]
#         sentence = sentence[1:]
#         # The first segment (0) is of images.
#         segment += 1

#         return sentence, segment, image, label



class LogFormatter:
    def __init__(self):
        self.start_time = time.time()

    def format(self, record):
        elapsed_seconds = round(record.created - self.start_time)

        prefix = "%s - %s - %s" % (
            record.levelname,
            time.strftime("%x %X"),
            timedelta(seconds=elapsed_seconds),
        )
        message = record.getMessage()
        message = message.replace("\n", "\n" + " " * (len(prefix) + 3))
        return "%s - %s" % (prefix, message)


def get_transforms():
    return transforms.Compose(
        [
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.46777044, 0.44531429, 0.40661017],
                std=[0.12221994, 0.12145835, 0.14380469],
            ),
        ]
    )


def collate_fn(batch):
    lens = [len(row[0]) for row in batch]
    bsz, max_seq_len = len(batch), max(lens)

    mask_tensor = torch.zeros(bsz, max_seq_len).long()
    text_tensor = torch.zeros(bsz, max_seq_len).long()
    segment_tensor = torch.zeros(bsz, max_seq_len).long()

    img_tensor = None
    if model_name in ["img", "concatbow", "concatbert", "mmbt"]:
        img_tensor = torch.stack([row[2] for row in batch])

    # Multilabel case
    tgt1_tensor = torch.stack([row[3] for row in batch])

    for i_batch, (input_row, length) in enumerate(zip(batch, lens)):
        tokens, segment = input_row[:2]
        text_tensor[i_batch, :length] = tokens
        segment_tensor[i_batch, :length] = segment
        mask_tensor[i_batch, :length] = 1

    return text_tensor, segment_tensor, mask_tensor, img_tensor, tgt1_tensor


def get_data_loaders():
    print("Loading data")
    transforms = get_transforms()

    # train = JsonlDataset(text_tr, img_tr, y1_tr, tokenizer, transforms)
    train = JsonlDataset(text_tr, web_ent_tr, img_tr, y1_tr, tokenizer, transforms)
    print(train)

    train_data_len = len(train)

    dev = JsonlDataset(text_val,web_ent_val,img_val, y1_val, tokenizer, transforms)
    # dev = JsonlDataset(text_val,img_val, y1_val, tokenizer, transforms)
    # print(dev)

    collate = functools.partial(collate_fn)

    train_loader = DataLoader(
        train,
        batch_size=batch_sz,
        shuffle=True,
        num_workers=n_workers,
        collate_fn=collate,
    )

    val_loader = DataLoader(
        dev,
        batch_size=batch_sz,
        shuffle=False,
        num_workers=n_workers,
        collate_fn=collate,
    )

    # test_set = JsonlDataset(text_test,img_test,y1_test, tokenizer, transforms)
    test_set = JsonlDataset(text_test,web_ent_test,img_test,y1_test, tokenizer, transforms)

    test_loader = DataLoader(
        test_set,
        batch_size=batch_sz,
        shuffle=False,
        num_workers=n_workers,
        collate_fn=collate,
    )

    return train_loader, val_loader, test_loader


def create_logger(filepath):
    # create log formatter
    log_formatter = LogFormatter()

    # create file handler and set level to debug
    file_handler = logging.FileHandler(filepath, "a")
    file_handler.setLevel(logging.DEBUG)
    file_handler.setFormatter(log_formatter)

    # create console handler and set level to info
    console_handler = logging.StreamHandler()
    console_handler.setLevel(logging.INFO)
    console_handler.setFormatter(log_formatter)

    # create logger and set level to debug
    logger = logging.getLogger()
    logger.handlers = []
    logger.setLevel(logging.DEBUG)
    logger.propagate = False
    logger.addHandler(file_handler)
    logger.addHandler(console_handler)

    # reset logger elapsed time
    def reset_time():
        log_formatter.start_time = time.time()

    logger.reset_time = reset_time

    logger.info(
        "\n".join(
            "%s: %s" % (k, str(v))
            for k, v in sorted(dict(vars()).items(), key=lambda x: x[0])
        )
    )
    return logger


def get_criterion():
      if weight_classes:
            freqs = [aspect_label_freqs[l] for l in aspect_label]
            label_weights = (torch.FloatTensor(freqs) / len(text_tr)) ** -1
            criterion = nn.BCEWithLogitsLoss(pos_weight=label_weights.cuda())
      else:
          criterion = nn.BCEWithLogitsLoss()
      return criterion

# def get_criterion():
#       criterion = AsymmetricLoss(gamma_neg=0, gamma_pos=1, clip=0.2, disable_torch_grad_focal_loss=True)
#       return criterion



def get_scheduler(optimizer):
    return optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, "max", patience=lr_patience, verbose=True, factor=lr_factor
    )

#### Modeling

In [ ]:
class RobertaEncoder(nn.Module):
    def __init__(self):
        super(RobertaEncoder, self).__init__()
        # self.args = args
        self.bert = RobertaModel.from_pretrained("roberta-base")

    def forward(self, txt, mask, segment):
        _, out = self.bert(
            txt,
            token_type_ids=segment,
            attention_mask=mask,
            output_all_encoded_layers=False,
        )
        # print(out.shape)
        return out

class ImageEncoder(nn.Module):
    def __init__(self):
        super(ImageEncoder, self).__init__()
        # self.args = args
        model = torchvision.models.resnet152(pretrained=True)
        modules = list(model.children())[:-2]
        self.model = nn.Sequential(*modules)

        pool_func = (
            nn.AdaptiveAvgPool2d
            if img_embed_pool_type == "avg"
            else nn.AdaptiveMaxPool2d
        )

        if num_image_embeds in [1, 2, 3, 5, 7]:
            self.pool = pool_func((num_image_embeds, 1))
        elif num_image_embeds == 4:
            self.pool = pool_func((2, 2))
        elif num_image_embeds == 6:
            self.pool = pool_func((3, 2))
        elif num_image_embeds == 8:
            self.pool = pool_func((4, 2))
        elif num_image_embeds == 9:
            self.pool = pool_func((3, 3))

    def forward(self, x):
        # Bx3x224x224 -> Bx2048x7x7 -> Bx2048xN -> BxNx2048
        out = self.pool(self.model(x))
        out = torch.flatten(out, start_dim=2)
        out = out.transpose(1, 2).contiguous()
        return out  # BxNx2048

class ImageBertEmbeddings(nn.Module):
    def __init__(self, embeddings):
        super(ImageBertEmbeddings, self).__init__()
        # self. =
        self.img_embeddings = nn.Linear(img_hidden_sz, hidden_sz)
        self.position_embeddings = embeddings.position_embeddings
        self.token_type_embeddings = embeddings.token_type_embeddings
        self.word_embeddings = embeddings.word_embeddings
        self.LayerNorm = embeddings.LayerNorm
        self.dropout = nn.Dropout(p=dropout)
        self.num_image_embeds = num_image_embeds
        # self.vocab = vocab
        self.tokenizer = tokenizer

    def forward(self, input_imgs, token_type_ids):
        bsz = input_imgs.size(0)
        seq_length = self.num_image_embeds + 2  # +2 for CLS and SEP Token

        cls_id = torch.LongTensor([self.tokenizer.convert_tokens_to_ids("[CLS]")]).cuda()
        cls_id = cls_id.unsqueeze(0).expand(bsz, 1)
        cls_token_embeds = self.word_embeddings(cls_id)

        sep_id = torch.LongTensor([self.tokenizer.convert_tokens_to_ids("[SEP]")]).cuda()
        sep_id = sep_id.unsqueeze(0).expand(bsz, 1)
        sep_token_embeds = self.word_embeddings(sep_id)

        imgs_embeddings = self.img_embeddings(input_imgs)
        token_embeddings = torch.cat(
            [cls_token_embeds, imgs_embeddings, sep_token_embeds], dim=1
        )

        position_ids = torch.arange(seq_length, dtype=torch.long).cuda()
        position_ids = position_ids.unsqueeze(0).expand(bsz, seq_length)
        position_embeddings = self.position_embeddings(position_ids)
        token_type_embeddings = self.token_type_embeddings(token_type_ids)
        embeddings = token_embeddings + position_embeddings + token_type_embeddings
        embeddings = self.LayerNorm(embeddings)
        embeddings = self.dropout(embeddings)
        return embeddings


class MultimodalRobertaEncoder(nn.Module):
    def __init__(self, ):
        super(MultimodalRobertaEncoder, self).__init__()
        bert = RobertaModel.from_pretrained("roberta-base")
        self.txt_embeddings = bert.embeddings

        ternary_embeds = nn.Embedding(3, hidden_sz)
        ternary_embeds.weight.data[:2].copy_(
        bert.embeddings.token_type_embeddings.weight
            )
        ternary_embeds.weight.data[2].copy_(
                bert.embeddings.token_type_embeddings.weight.data.mean(dim=0)
            )
        self.txt_embeddings.token_type_embeddings = ternary_embeds

        self.img_embeddings = ImageBertEmbeddings(self.txt_embeddings)
        self.img_encoder = ImageEncoder()
        self.encoder = bert.encoder
        self.pooler = bert.pooler
        self.num_image_embeds = num_image_embeds

    def forward(self, input_txt, attention_mask, segment, input_img):
        bsz = input_txt.size(0)
        attention_mask = torch.cat(
            [
                torch.ones(bsz, self.num_image_embeds + 2).long().cuda(),
                attention_mask,
            ],
            dim=1,
        )
        extended_attention_mask = attention_mask.unsqueeze(1).unsqueeze(2)
        extended_attention_mask = extended_attention_mask.to(
            dtype=next(self.parameters()).dtype
        )
        extended_attention_mask = (1.0 - extended_attention_mask) * -10000.0

        img_tok = (
            torch.LongTensor(input_txt.size(0), self.num_image_embeds + 2)
            .fill_(0)
            .cuda()
        )
        img = self.img_encoder(input_img)  # BxNx3x224x224 -> BxNx2048
        img_embed_out = self.img_embeddings(img, img_tok)
        txt_embed_out = self.txt_embeddings(input_txt, segment)
        encoder_input = torch.cat([img_embed_out, txt_embed_out], 1)  # Bx(TEXT+IMG)xHID

        encoded_layers = self.encoder(
            encoder_input, extended_attention_mask)#, output_all_encoded_layers=False)

        return self.pooler(encoded_layers[-1])


class MultimodalBertClf(nn.Module):
    def __init__(self, ):
        super(MultimodalBertClf, self).__init__()

        self.enc = MultimodalRobertaEncoder()
        self.clf1 = nn.Linear(hidden_sz, n_classes)

    def forward(self, txt, mask, segment, img):
        x = self.enc(txt, mask, segment, img)
        out_head1 = self.clf1(x)
        return (out_head1)

#### Forward method

In [ ]:
def model_forward(i_epoch, model, criterion, batch):
    txt, segment, mask, img, tgt1 = batch

    for param in model.enc.img_encoder.parameters():
        param.requires_grad = not freeze_img
    for param in model.enc.encoder.parameters():
        param.requires_grad = not freeze_txt

    txt, img = txt.cuda(), img.cuda()
    mask, segment = mask.cuda(), segment.cuda()
    # print(mask.shape)
    # print(segment.shape)

    out1 = model(txt, mask, segment, img)

    tgt1 = tgt1.cuda()
    # print(tgt1.shape)
    # print(out1.shape)
    loss1 = criterion(out1, tgt1)


    return loss1, out1, tgt1

#### Evaluate and predict

In [ ]:
def model_eval(i_epoch, data, model, criterion, store_preds=False):
    with torch.no_grad():
        losses, preds1, tgts1 = [], [], []
        for batch in data:
            loss, out1, tgt1 = model_forward(i_epoch, model, criterion, batch)
            losses.append(loss.item())

            pred1 = torch.sigmoid(out1).cpu().detach().numpy() > 0.5

            preds1.append(pred1)

            tgt1 = tgt1.cpu().detach().numpy()
            tgts1.append(tgt1)

    metrics = {"loss_mean_overall": np.mean(losses)}
    tgts1 = np.vstack(tgts1)
    preds1 = np.vstack(preds1)
    metrics["macro_f1"] = f1_score(tgts1, preds1, average="macro")
    metrics["micro_f1"] = f1_score(tgts1, preds1, average="micro")

    return metrics


def model_pred(i_epoch, data, model, criterion, store_preds=False):
    with torch.no_grad():
        losses, preds1, tgts1 = [], [], []
        for batch in data:
            loss, out1, tgt1 = model_forward(i_epoch, model, criterion, batch)
            losses.append(loss.item())

            pred1 = torch.sigmoid(out1).cpu().detach().numpy() > 0.5
            preds1.append(pred1)

            tgt1 = tgt1.cpu().detach().numpy()
            tgts1.append(tgt1)

        return (tgts1, preds1)

## Initiating model

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

set_seed(seed)
print(os.path)
savedir = os.path.join(savedir, name)
print(savedir)
os.makedirs(savedir, exist_ok=True)

train_loader, val_loader, test_loaders = get_data_loaders()

model = MultimodalBertClf()
criterion = get_criterion()
optimizer = optim.Adam(model.parameters(), lr=lr)
scheduler = get_scheduler(optimizer)

logger = create_logger("%s/logfile.log" % savedir)
logger.info(model)
model.cuda()

start_epoch, global_step, n_no_improve, best_metric = 0, 0, 0, -np.inf

if os.path.exists(os.path.join(savedir, "checkpoint.pt")):
        checkpoint = torch.load(os.path.join(savedir, "checkpoint.pt"))
        start_epoch = checkpoint["epoch"]
        n_no_improve = checkpoint["n_no_improve"]
        best_metric = checkpoint["best_metric"]
        model.load_state_dict(checkpoint["state_dict"])
        optimizer.load_state_dict(checkpoint["optimizer"])
        scheduler.load_state_dict(checkpoint["scheduler"])

## Training

In [ ]:
torch.cuda.empty_cache()
logger.info("Training..")
for i_epoch in range(start_epoch, max_epochs):
        train_losses = []
        model.train()
        optimizer.zero_grad()
        print(train_loader)
        for batch in tqdm(train_loader):
            loss, _, _ = model_forward(i_epoch, model,criterion, batch)
            if gradient_accumulation_steps > 1:
                loss = loss / gradient_accumulation_steps

            train_losses.append(loss.item())
            loss.backward()
            global_step += 1
            if global_step % gradient_accumulation_steps == 0:
                optimizer.step()
                optimizer.zero_grad()

        model.eval()
        metrics = model_eval(i_epoch, val_loader, model, criterion)
        logger.info("Train Loss: {:.4f}".format(np.mean(train_losses)))
        log_metrics("Val", metrics,logger)

        tuning_metric = metrics["micro_f1"]

        scheduler.step(tuning_metric)
        is_improvement = tuning_metric > best_metric
        if is_improvement:
            best_metric = tuning_metric
            n_no_improve = 0
        else:
            n_no_improve += 1

        save_checkpoint(
            {
                "epoch": i_epoch + 1,
                "state_dict": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
                "n_no_improve": n_no_improve,
                "best_metric": best_metric,
            },
            is_improvement,
            savedir,
        )

        if n_no_improve >= patience:
            logger.info("No improvement. Breaking out of loop.")
            break

## Testing

In [ ]:
load_checkpoint(model, "/content/drive/MyDrive/harsha_451/test/final_test/nameless/model_best.pt")
model.eval()
test_metrics = model_eval(
            np.inf, test_loaders, model, criterion, store_preds=False
        )

print('test_metrics')
print()
test_metrics

In [ ]:
# text_test[0]
# print(text_test[0])
# temp_str = text_test[0].split(' ',3)
# #print(temp_str)
# temp_credit = temp_str[0]+" "+temp_str[1]
# print(temp_credit)
# new_text = 0
# # for text in text_review:
# #   temp = text.split(' ',3)
# #   temp_customer = temp[0]+ " "+temp[1]
# #   if temp_customer == 'credit card':
# #     new_text = new_text + 1

# # print(new_text)

In [ ]:
load_checkpoint(model, "/content/drive/MyDrive/harsha_451/test/final_test/nameless/model_best.pt")
model.eval()

collate = functools.partial(collate_fn)
# transforms = get_transforms()
#print(aspect_label)
#len(y1_test)
domain_list = ['Transactional Deficiency','Investments','Debit Card','Economy','Customer Service','Credit card','Loan','Financial Policies','Retail banking','Salary']


new_text_test = []
new_web_ent_test = []
new_img_test = []
new_y1_test = []
count = 0
for i in range(len(text_review)):
  #text_test[i]
  #print(text_test[i])
  temp_str = text_review[i].split(' ',3)
  #print(temp_str)
  # temp_credit = temp_str[0]+" "+temp_str[1]
  temp_credit = temp_str[0]
  #print(temp_credit)
  if(temp_credit == 'salary'):
      new_text_test.append(text_review[i])
      new_web_ent_test.append(web_entities[i])
      new_img_test.append(images_path_new[i])
      new_y1_test.append(aspect_labels[i])
      count = count + 1

print(count)
transforms_new = get_transforms()
test_set = JsonlDataset(new_text_test,new_web_ent_test,new_img_test,new_y1_test, tokenizer,transforms_new)
test_loader = DataLoader(
    test_set,
    batch_size=batch_sz,
    shuffle=False,
    num_workers=n_workers,
    collate_fn=collate,
)
test_metrics = model_eval(
        np.inf, test_loader, model, criterion, store_preds=False
    )
print('Salary\n')
print(f"{test_metrics}\n")


In [ ]:
load_checkpoint(model, "/content/drive/MyDrive/harsha_451/test/final_test/nameless/model_best.pt")
model.eval()
collate = functools.partial(collate_fn)
transforms_new = get_transforms()
new_text_10 = text_review[slice(30)]
new_web_ent_10 = web_entities[slice(30)]
new_img_test_10 = images_path_new[slice(30)]
new_y1_test_10 = aspect_labels[slice(30)]
test_set = JsonlDataset(new_text_10,new_web_ent_10,new_img_test_10,new_y1_test_10, tokenizer,transforms_new)
test_loader = DataLoader(
    test_set,
    batch_size=30,
    shuffle=False,
    num_workers=n_workers,
    collate_fn=collate,
)
target,predicts = model_pred(
        np.inf, test_loader, model, criterion, store_preds=False
    )

In [ ]:
print(len(predicts))

In [ ]:
print((target))

In [ ]:
from sklearn.metrics import multilabel_confusion_matrix

# Assuming you have your targets and predictions as described
# Replace `targets` and `predictions` with your actual variables

# Initialize an empty list to store confusion matrices for each class
confusion_matrices = []

# Calculate the confusion matrix for each class
for i in range(len(target)):
    confusion_matrix = multilabel_confusion_matrix(target[i], predicts[i])
    confusion_matrices.append(confusion_matrix)

# Now `confusion_matrices` contains the confusion matrices for each class
# You can access each confusion matrix using indexing, e.g., confusion_matrices[0] for the first class



In [ ]:
print(aspect_label)

In [ ]:
print(len(confusion_matrices[0]))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Assuming you have `confusion_matrices` as described previously

# Define the class names (replace with your actual class names)
class_names = aspect_label  # Replace with your class names

# Loop through each class's confusion matrix
for i, confusion_matrix in enumerate(confusion_matrices[0]):
    class_name = class_names[i]

    # Create a figure and axis
    plt.figure(figsize=(6, 4))

    # Create a heatmap of the confusion matrix
    sns.heatmap(confusion_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=['Negative', 'Positive'], yticklabels=['Negative', 'Positive'])

    # Set axis labels and title
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title(f'Confusion Matrix for Class: {class_name}')

    # Display the plot
    plt.show()


In [ ]:


# Assuming you have a list of confusion matrices named `confusion_matrices`

# Initialize an empty aggregated confusion matrix
aggregated_confusion_matrix = np.zeros((2, 2), dtype=int)  # Assuming binary classification

# Sum up the individual confusion matrices
for confusion_matrix in confusion_matrices[0]:
    aggregated_confusion_matrix += confusion_matrix

# Display the aggregated confusion matrix
print("Aggregated Confusion Matrix:")
print(aggregated_confusion_matrix)
